# Baseline рекомендательной системы

На этом этапе построим простые рекомендательные алгоритмы, которые будут использоваться как точка отсчёта для более сложных методов.

Рассмотрим два baseline:

1. **Popular baseline** — рекомендация наиболее популярных товаров.
2. **Recent Items baseline** — рекомендация последних уникальных товаров из истории текущей сессии.

Качество будем оценивать с помощью **Recall@20**, **Precision@20** и **NDCG@20** отдельно для `clicks`, `carts` и `orders`.

Данные из 'history' используем для построения рекомендаций, из 'future' - для оценки качества модели.

In [15]:
import pandas as pd 
from pathlib import Path 

data_path = Path("../data/otto-recsys-train.jsonl")

df = pd.read_json(
    data_path,
    lines=True,
    nrows=10000
)

df.head()

,session,events
0,0,"[{'aid': 1517085, 'ts': 1659304800025, 'type':..."
1,1,"[{'aid': 424964, 'ts': 1659304800025, 'type': ..."
2,2,"[{'aid': 763743, 'ts': 1659304800038, 'type': ..."
3,3,"[{'aid': 1425967, 'ts': 1659304800095, 'type':..."
4,4,"[{'aid': 613619, 'ts': 1659304800119, 'type': ..."


## Подготовка history и future

Каждая сессия будет разделена на две части:

- первые 75% событий — 'history', доступная рекомендательной системе;
- последние 25% событий — 'future', используемая только для оценки рекомендаций.

In [16]:
history_arr = []
future_arr = []

for events in df['events']:
    future_size = max(1, int(len(events) * 0.25))

    history = events[:-future_size]
    future = events[-future_size:]

    history_arr.append(history)
    future_arr.append(future)

df['history'] = history_arr
df['future'] = future_arr


df[['session', 'history', 'future']].head()

,session,history,future
0,0,"[{'aid': 1517085, 'ts': 1659304800025, 'type':...","[{'aid': 362233, 'ts': 1661551907312, 'type': ..."
1,1,"[{'aid': 424964, 'ts': 1659304800025, 'type': ...","[{'aid': 959544, 'ts': 1659991199405, 'type': ..."
2,2,"[{'aid': 763743, 'ts': 1659304800038, 'type': ...","[{'aid': 295985, 'ts': 1661713601647, 'type': ..."
3,3,"[{'aid': 1425967, 'ts': 1659304800095, 'type':...","[{'aid': 88711, 'ts': 1659998456613, 'type': '..."
4,4,"[{'aid': 613619, 'ts': 1659304800119, 'type': ...","[{'aid': 758750, 'ts': 1661504510200, 'type': ..."


## Popular baseline

Для первого baseline выберем рекомендацию самых популярных товаров.

Популярность определим как общее количество взаимодействий во всех сессиях 'history'. Вес типов взаимодействия (`clicks`, `carts`, `orders`) пока учитывать не будем.

In [17]:
item_counts = {}

for history in df['history']:
    for event in history:
        if event['aid'] not in item_counts:
            item_counts[event['aid']] = 0
            
        item_counts[event['aid']] += 1

popular_items = sorted(
    item_counts,
    key=item_counts.get,
    reverse=True
)

top_20_popular = popular_items[:20]

top_20_popular

[29735,
 832192,
 108125,
 1743151,
 1733943,
 1083665,
 1498443,
 80222,
 1658239,
 332654,
 612920,
 554660,
 476629,
 166037,
 1684953,
 247240,
 673407,
 1460571,
 959208,
 1502122]

Получен список 20 наиболее популярных товаров на основе событий из 'history'.

## Оценка baseline

Для оценки рекомендаций сформируем множества релевантных товаров из 'future' отдельно для `clicks`, `carts` и `orders`.

После этого сравним один и тот же список `top_20_popular` с реальными будущими действиями каждой сессии.

In [18]:
clicks_arr = []
carts_arr = []
orders_arr = []

for future in df['future']:
    clicks = set()
    carts = set()
    orders = set()

    for event in future:
        if event['type'] == 'clicks':
            clicks.add(event['aid'])
        elif event['type'] == 'carts':
            carts.add(event['aid'])
        elif event['type'] == 'orders':
            orders.add(event['aid'])

    clicks_arr.append(clicks)
    carts_arr.append(carts)
    orders_arr.append(orders)

df['future_clicks'] = clicks_arr
df['future_carts'] = carts_arr
df['future_orders'] = orders_arr

Импортируем функции метрик из ноутбука `02_validation.ipynb`.

In [19]:
import math

def recall_at_k(recommend, relevant_items, k=20):
    if len(relevant_items) == 0:
        return None

    recommend = recommend[:k]

    hits = 0
    for item in recommend:
        if item in relevant_items:
            hits += 1

    return hits / len(relevant_items)


def precision_at_k(recommend, relevant_items, k=20):
    recommend = recommend[:k]

    hits = 0
    for item in recommend:
        if item in relevant_items:
            hits += 1

    return hits / k


def ndcg_at_k(recommend, relevant_items, k=20):
    if len(relevant_items) == 0:
        return None

    recommend = recommend[:k]

    dcg = 0
    for pos, item in enumerate(recommend, start=1):
        if item in relevant_items:
            dcg += 1 / math.log2(pos + 1)

    ideal_hits = min(len(relevant_items), k)

    idcg = 0
    for pos in range(1, ideal_hits + 1):
        idcg += 1 / math.log2(pos + 1)

    return dcg / idcg

### Расчёт метрик

Для каждого типа событий метрики расчитываются только по сессиям, где в 'future' есть хотя бы один товар данного типа.

Это позволяет оценивать качество ранжирования отдельно от вероятности возникновения самого события.

In [20]:
def ev_baseline(recommend, targets, k=20):
    recalls = []
    precisions = []
    ndcgs = []

    for relevant_items in targets:
        if len(relevant_items) == 0:
            continue

        recalls.append(
            recall_at_k(recommend, relevant_items, k)
        )
        precisions.append(
            precision_at_k(recommend, relevant_items, k)
        )
        ndcgs.append(
            ndcg_at_k(recommend, relevant_items, k)
        )

    return {
        'Recall@20': sum(recalls) / len(recalls),
        'Precision@20': sum(precisions) / len(precisions),
        'NDCG@20': sum(ndcgs) / len(ndcgs)
    }


click_metrics = ev_baseline(
    top_20_popular,
    df['future_clicks']
)

cart_metrics = ev_baseline(
    top_20_popular,
    df['future_carts']
)

order_metrics = ev_baseline(
    top_20_popular,
    df['future_orders']
)

print('Clicks:', click_metrics)
print('Carts:', cart_metrics)
print('Orders:', order_metrics)

Clicks: {'Recall@20': 0.008167432217729165, 'Precision@20': 0.0030571892111133907, 'NDCG@20': 0.0055013168604016935}
Carts: {'Recall@20': 0.007559669344510419, 'Precision@20': 0.0007946210268948652, 'NDCG@20': 0.0034922725946699117}
Orders: {'Recall@20': 0.006325878594249201, 'Precision@20': 0.0005111821086261982, 'NDCG@20': 0.0022342415553655535}


### Результаты Popular baseline

Данный baseline показал низкое качество рекомендаций.

- **Recall@20** составляет менее 1% для всех типов событий.
- Наибольшее качество наблюдается для `clicks`, а наименьшее — для `orders`.
- Низкие значения **NDCG@20** показывают, что глобальная популярность плохо отражает намерения конкретной сессии.

Это было ожидаемо, потому что всем сессиям рекомендуется набор из 20 товаров, при том не учитываются индвидуальные предпочтения и история пользователя.

## Recent Items baseline

Во втором baseline будет использоваться история одной сессии, а именно: будут рекомендоваться товары, с которыми пользователь ранее взаимодейстовал в 'history'. Рекомендации будут ранжироваться в зависимости от недавней активности.

P.S. Проблемы с сессиями, в которых нет взаимодействий с товарами, быть не должно, так как из EDA мы выяснили, что минимальная длина сессии в 'history' - 2 события.

In [21]:
def recent_items(history, k=20):
    recommend = []
    seen = set()

    for event in reversed(history):
        curr_aid = event['aid']

        if curr_aid not in seen:
            recommend.append(curr_aid)
            seen.add(curr_aid)

        if len(recommend) == k:
            break
            
    return recommend

Проверим на первой сессии.

In [22]:
recent_items(df['history'].iloc[0], k=20)

[661144,
 570955,
 1760145,
 974651,
 275288,
 543308,
 138431,
 1157411,
 190818,
 102416,
 1436439,
 738987,
 1624436,
 1639229,
 1521766,
 280978,
 1766089,
 7563,
 1433235,
 6851]

### Проверка количества рекомендаций

`Recent Items` может вернуть меньше 20 рекомендаций, если в истории сессии
содержится меньше 20 уникальных товаров.

Проверим, для какой доли сессий baseline не способен сформировать полный топ-20.

In [23]:
recommend_lengths = []

for history in df['history']:
    recommend = recent_items(history, k=20)
    recommend_lengths.append(len(recommend))

less_than_20 = 0

for length in recommend_lengths:
    if length < 20:
        less_than_20 += 1

print('Сессий с < 20 рекомендациями:', less_than_20)
print('Доля таких сессий:', less_than_20 / len(df))

Сессий с < 20 рекомендациями: 6023
Доля таких сессий: 0.6023


### Popularity fallback для коротких сессий

В 60.23% сессий `Recent Items` формирует менее 20 рекомендаций, поскольку
в 'history' содержится недостаточно уникальных товаров.

Поэтому недостающие позиции будем заполнять глобально популярными товарами.

Итоговая логика baseline:

1. товары из текущей сессии ранжируются от наиболее недавних к более ранним;
2. повторяющиеся товары исключаются;
3. если получено меньше 20 рекомендаций, список дополняется популярными товарами;
4. уже добавленные товары повторно не добавляются.

Такой подход сохраняет персонализацию по текущей сессии и одновременно
позволяет сформировать полный топ-20.

In [24]:
def recent_items_with_fallback(history, popular_items, k=20):
    recommend = recent_items(history, k)

    if len(recommend) < k:
        seen = set(recommend)

        for aid in popular_items:
            if aid not in seen:
                recommend.append(aid)
                seen.add(aid)

            if len(recommend) == k:
                break

    return recommend

In [25]:
for history in df['history']:
    if len(recent_items(history, k=20)) < 20:
        short_history = history
        break

print('До fallback:', len(recent_items(short_history, k=20)))

recommend = recent_items_with_fallback(
    short_history,
    popular_items,
    k=20
)

print('После fallback:', len(recommend))
print(recommend)

До fallback: 15
После fallback: 20
[105393, 50049, 711125, 718983, 215311, 854637, 1734061, 1507622, 109488, 440486, 1515526, 424964, 1491172, 910862, 1492293, 29735, 832192, 108125, 1743151, 1733943]


## Оценка Recent Items baseline

Теперь оценим качество `Recent Items` с `Popularity fallback`.

В отличие от глобального `Popularity baseline`, рекомендации формируются отдельно
для каждой сессии на основе её истории.

Метрики `Recall@20`, `Precision@20` и `NDCG@20` считаются отдельно для
`clicks`, `carts` и `orders`.

Сессии, в которых в `future` отсутствует соответствующий тип события,
не учитываются при расчёте метрик для этого типа.

In [26]:
def evaluate_recent_items(histories, targets, popular_items, k=20):
    recalls = []
    precisions = []
    ndcgs = []

    for history, relevant_items in zip(histories, targets):
        if len(relevant_items) == 0:
            continue

        recommend = recent_items_with_fallback(
            history,
            popular_items,
            k
        )

        recalls.append(
            recall_at_k(recommend, relevant_items, k)
        )

        precisions.append(
            precision_at_k(recommend, relevant_items, k)
        )

        ndcgs.append(
            ndcg_at_k(recommend, relevant_items, k)
        )

    return {
        'Recall@20': sum(recalls) / len(recalls),
        'Precision@20': sum(precisions) / len(precisions),
        'NDCG@20': sum(ndcgs) / len(ndcgs)
    }

In [27]:
recent_clicks = evaluate_recent_items(
    df['history'],
    df['future_clicks'],
    popular_items
)

recent_carts = evaluate_recent_items(
    df['history'],
    df['future_carts'],
    popular_items
)

recent_orders = evaluate_recent_items(
    df['history'],
    df['future_orders'],
    popular_items
)

print('Clicks:', recent_clicks)
print('Carts:', recent_carts)
print('Orders:', recent_orders)

Clicks: {'Recall@20': 0.22785842502171122, 'Precision@20': 0.06419590346785624, 'NDCG@20': 0.2304360725285293}
Carts: {'Recall@20': 0.2792700395413184, 'Precision@20': 0.028927261613691274, 'NDCG@20': 0.22352544122505233}
Orders: {'Recall@20': 0.48686814401129047, 'Precision@20': 0.055271565495206905, 'NDCG@20': 0.35758412893856895}


## Сравнение baseline-моделей

Сравним два простых подхода:

- `Popularity` — одинаковые глобально популярные товары для всех сессий;
- `Recent Items + Popularity fallback` — последние уникальные товары конкретной
  сессии с дополнением популярными товарами.

Это позволит оценить, насколько использование истории текущей сессии улучшает
качество рекомендаций по сравнению с неперсонализированным подходом.

In [30]:
import pandas as pd

results = pd.DataFrame([
    ['Popularity', 'Clicks', 0.00817, 0.00306, 0.00550],
    ['Popularity', 'Carts', 0.00756, 0.00079, 0.00349],
    ['Popularity', 'Orders', 0.00633, 0.00051, 0.00223],

    ['Recent Items + fallback', 'Clicks',
     recent_clicks['Recall@20'],
     recent_clicks['Precision@20'],
     recent_clicks['NDCG@20']],

    
    ['Recent Items + fallback', 'Carts',
     recent_carts['Recall@20'],
     recent_carts['Precision@20'],
     recent_carts['NDCG@20']],

    
    ['Recent Items + fallback', 'Orders',
     recent_orders['Recall@20'],
     recent_orders['Precision@20'],
     recent_orders['NDCG@20']]
], columns=[
    'Baseline',
    'Target',
    'Recall@20',
    'Precision@20',
    'NDCG@20'
])

results.round(4)

,Baseline,Target,Recall@20,Precision@20,NDCG@20
0,Popularity,Clicks,0.0082,0.0031,0.0055
1,Popularity,Carts,0.0076,0.0008,0.0035
2,Popularity,Orders,0.0063,0.0005,0.0022
3,Recent Items + fallback,Clicks,0.2279,0.0642,0.2304
4,Recent Items + fallback,Carts,0.2793,0.0289,0.2235
5,Recent Items + fallback,Orders,0.4869,0.0553,0.3576


## Вывод

`Recent Items + Popularity fallback` значительно превосходит глобальный `Popularity baseline` по всем рассматриваемым типам событий.

Это еще раз доказывает, что история текущей сессии содержит ценную информацию, используемую для предсказания будущих действий пользователя.

Особенная разница наблюдается в `Recall@20` для `orders`: значительная часть товаров, которые пользователь покупает в будущем, уже встречаются в истории его сессий (то есть в истории заказов).

При этом baseline ограничен товарами, которые уже присутствовали в истории, и небольшим набором популярных товаров. Он плохо подходит для поиска новых релевантных товаров, которых пользователь ещё не видел.

Следующим этапом проекта будет генерация кандидатов. Цель этого этапа - расширить набор потенциально релевантных товаров, сохранив текущие метрики (Recall), после чего кандидаты будут ранжироваться более сложной моделью.